<div style="font-size: 15.5px; line-height: 1.6; background: #ffffff; padding: 20px 28px; border-radius: 8px; color: #1a365d;">

<h2 style="font-size: 24px; margin: 4px 0 16px 0; color: #1a365d; border-bottom: 2px solid #3182ce; padding-bottom: 8px;">Module 12 · Notebook 01 — Self-RAG</h2>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">The Problem with Basic RAG</h3>

<p style="margin: 10px 0; color: #1a365d;">Every RAG pipeline you've built so far <strong>always retrieves</strong>. When a user asks <em>"What is cassava mosaic disease?"</em>, we retrieve. When they ask <em>"What is 2 + 2?"</em> — we <strong>still retrieve</strong>. That's wasteful and often wrong.</p>

<table style="font-size: 15px; margin: 10px 0; border-collapse: collapse; width: 100%; background: #ffffff; box-shadow: 0 1px 3px rgba(0,0,0,0.06); color: #1a365d;">
  <thead>
    <tr style="background: #ebf4ff;">
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">User question</th>
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Basic RAG behaviour</th>
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">What should happen</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">"What is cassava mosaic disease?"</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Retrieve ✅</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Retrieve ✅</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">"What is 2 + 2?"</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Retrieve ❌</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Answer directly</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">"Hello, how are you?"</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Retrieve ❌</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Answer directly</td></tr>
    <tr><td style="padding: 10px 14px; color: #1a365d;">"What is the capital of France?"</td><td style="padding: 10px 14px; color: #1a365d;">Retrieve ❌</td><td style="padding: 10px 14px; color: #1a365d;">Answer directly (or admit)</td></tr>
  </tbody>
</table>

<p style="margin: 10px 0; color: #1a365d;">Every unnecessary retrieval:</p>

<ul style="margin: 10px 0; padding-left: 26px; color: #1a365d;">
  <li>Costs an embedding call + a vector search</li>
  <li>Adds latency</li>
  <li>Can <strong>confuse the LLM</strong> with irrelevant context</li>
  <li>Wastes tokens in the final prompt</li>
</ul>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">What Self-RAG Does</h3>

<p style="margin: 10px 0; color: #1a365d;"><strong>Self-RAG</strong> adds a <strong>decision node</strong> before retrieval:</p>

<pre style="font-size: 14px; line-height: 1.5; margin: 10px 0; padding: 14px 18px; background: #ebf4ff; color: #1a365d; border-radius: 6px; border-left: 4px solid #3182ce; overflow-x: auto;"><code>User question
     │
     ▼
[Decide: does this need retrieval?]
     │
     ├── No  → Answer directly with LLM
     │
     └── Yes → Retrieve → Rerank → Answer</code></pre>

<p style="margin: 10px 0; color: #1a365d;">The decision is made by an <strong>LLM classifier</strong> — same technique we used for LIST vs FACT routing in the context compression notebook.</p>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">Why "Self"-RAG?</h3>

<p style="margin: 10px 0; color: #1a365d;">Because the system <strong>reflects on itself</strong> before acting:</p>

<ul style="margin: 10px 0; padding-left: 26px; color: #1a365d;">
  <li><strong>Self-decides</strong> whether to retrieve</li>
  <li>Later (Corrective RAG) it will <strong>self-grade</strong> whether retrieval was good</li>
  <li>Later still it will <strong>self-correct</strong> by retrying</li>
</ul>

<p style="margin: 10px 0; color: #1a365d;">Self-RAG is the first step: <strong>self-decide</strong>.</p>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">How the Decision is Made</h3>

<p style="margin: 10px 0; color: #1a365d;">We use a strict classification prompt:</p>

<pre style="font-size: 14px; line-height: 1.5; margin: 10px 0; padding: 14px 18px; background: #ebf4ff; color: #1a365d; border-radius: 6px; border-left: 4px solid #3182ce; overflow-x: auto;"><code>System: You decide if a question needs the knowledge base.
        Categories:
        - RETRIEVE: about crops, diseases, treatments, Nigeria health facts
        - DIRECT:   general knowledge, greetings, math, chit-chat
        Respond with exactly one word: RETRIEVE or DIRECT.</code></pre>

<p style="margin: 10px 0; color: #1a365d;">Then the agent routes based on the answer.</p>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">Self-RAG vs Basic RAG</h3>

<table style="font-size: 15px; margin: 10px 0; border-collapse: collapse; width: 100%; background: #ffffff; box-shadow: 0 1px 3px rgba(0,0,0,0.06); color: #1a365d;">
  <thead>
    <tr style="background: #ebf4ff;">
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Aspect</th>
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Basic RAG</th>
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Self-RAG</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Always retrieves?</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Yes</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Only when needed</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">LLM calls per query</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">1 (answer)</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">2 (classify + answer)</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Latency on chit-chat</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Slow (unnecessary retrieve)</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Fast (skip retrieve)</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Context pollution</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">High</td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Low</td></tr>
    <tr><td style="padding: 10px 14px; color: #1a365d;">Cost on chit-chat</td><td style="padding: 10px 14px; color: #1a365d;">High</td><td style="padding: 10px 14px; color: #1a365d;">Low</td></tr>
  </tbody>
</table>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">Why This Matters for AgriVoice</h3>

<p style="margin: 10px 0; color: #1a365d;">Farmers will ask all kinds of things:</p>

<ul style="margin: 10px 0; padding-left: 26px; color: #1a365d;">
  <li>"Hello" → <strong>direct answer</strong></li>
  <li>"What's the weather?" → <strong>direct (calls weather tool)</strong></li>
  <li>"How do I treat cassava mosaic?" → <strong>retrieve</strong></li>
  <li>"Thank you" → <strong>direct answer</strong></li>
  <li>"What is the price of maize in Kano?" → <strong>retrieve</strong></li>
</ul>

<p style="margin: 10px 0; color: #1a365d;">Without Self-RAG, every single one of those triggers a vector search — costing money, adding latency, and giving the LLM noise to filter through.</p>

<p style="margin: 10px 0; color: #1a365d;">With Self-RAG, only the ones that actually need knowledge retrieval pay for it.</p>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">What We'll Build</h3>

<table style="font-size: 15px; margin: 10px 0; border-collapse: collapse; width: 100%; background: #ffffff; box-shadow: 0 1px 3px rgba(0,0,0,0.06); color: #1a365d;">
  <thead>
    <tr style="background: #ebf4ff;">
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Cell</th>
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">What it does</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>1</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Imports and setup</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>2</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Load the vector store (chroma_db_domain)</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>3</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">The classifier prompt — RETRIEVE vs DIRECT</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>4</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">The classifier chain</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>5</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Direct answer chain (no context)</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>6</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Retrieval answer chain (with context)</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>7</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">The self-RAG router</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>8</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">Full `ask()` function with routing</td></tr>
    <tr><td style="padding: 10px 14px; color: #1a365d;"><strong>9</strong></td><td style="padding: 10px 14px; color: #1a365d;">Test suite — 6 different question types</td></tr>
  </tbody>
</table>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">Key Terms</h3>

<table style="font-size: 15px; margin: 10px 0; border-collapse: collapse; width: 100%; background: #ffffff; box-shadow: 0 1px 3px rgba(0,0,0,0.06); color: #1a365d;">
  <thead>
    <tr style="background: #ebf4ff;">
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Term</th>
      <th style="text-align: left; padding: 10px 14px; border-bottom: 2px solid #bee3f8; color: #1a365d;">Meaning</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>Self-RAG</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">RAG that decides whether retrieval is needed</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>Router</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">The classifier that picks RETRIEVE or DIRECT</td></tr>
    <tr><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;"><strong>Direct path</strong></td><td style="padding: 10px 14px; border-bottom: 1px solid #e2e8f0; color: #1a365d;">LLM answers without retrieval</td></tr>
    <tr><td style="padding: 10px 14px; color: #1a365d;"><strong>Retrieval path</strong></td><td style="padding: 10px 14px; color: #1a365d;">Retrieve → rerank → answer</td></tr>
  </tbody>
</table>

<h3 style="font-size: 19px; margin: 22px 0 10px 0; color: #2c5282;">What Comes Next</h3>

<p style="margin: 10px 0; color: #1a365d;">Once Self-RAG works, we'll extend it:</p>

<ul style="margin: 10px 0; padding-left: 26px; color: #1a365d;">
  <li><strong>Notebook 02 — Corrective RAG:</strong> after retrieving, <em>grade</em> the chunks. If they're bad, retry with a different query or fall back to no-retrieval mode.</li>
  <li><strong>Notebook 03 — Adaptive RAG:</strong> route to different retrievers (crops vs health vs livestock).</li>
  <li><strong>Notebook 04 — Agentic GraphRAG:</strong> use graph traversal as one more tool the agent can call.</li>
</ul>

<p style="margin: 10px 0; color: #1a365d;">Each is a small addition to what we build today.</p>

</div>